# ST422 Brief 8 — Quality Assurance: Data Load Check

**Author:** Akeel Shah | **Student ID:** u2211111  
**Date:** April 2026  
**Purpose:** Verify that `ST422_DataPrep.ipynb` produced correct, complete, and consistent output files.

This notebook does **not** rerun the data preparation. It loads the four output files from the `Cleaned` directory and runs a series of checks to confirm they are valid and ready for analysis.

**Run order:** This notebook must be run **after** `ST422_DataPrep.ipynb` has been run successfully.

---
### Checks performed
1. All output files exist
2. Row counts are plausible
3. Year range is correct
4. No duplicate rows
5. Key columns present and not fully missing
6. Severity codes are valid
7. Join integrity — cas_full matches casualties table
8. Geographic coverage — LAs present
9. Summary pass/fail report

---
## Setup

In [1]:
import pandas as pd
import numpy as np
import os

# ── USER CONFIG ──────────────────────────────────────────────────────────────
# Update this to match your machine — same as OUTPUT_DIR in ST422_DataPrep.ipynb
CLEANED_DIR = 'C:/Users/u2211111/ST422/Cleaned'
# ─────────────────────────────────────────────────────────────────────────────

# Expected files
EXPECTED_FILES = [
    'cas_full.csv',
    'casualties_clean.csv',
    'collisions_clean.csv',
    'vehicles_clean.csv'
]

# QA results tracker
qa_results = []

def record(check, passed, detail):
    status = ' PASS' if passed else ' FAIL'
    qa_results.append({'Check': check, 'Status': status, 'Detail': detail})
    print(f'{status} | {check}: {detail}')

print('Setup complete. Running QA checks...')
print(f'Checking files in: {CLEANED_DIR}')
print()

Setup complete. Running QA checks...
Checking files in: C:/Users/u2211111/ST422/Cleaned



---
## Check 1: All output files exist
Confirms that all four expected output files were created by `ST422_DataPrep.ipynb` and are present in the `Cleaned` directory. Also reports file size as a basic sanity check — a file that is unexpectedly small may indicate a failed or incomplete write.

In [2]:
print('=== CHECK 1: File existence ===')
for fname in EXPECTED_FILES:
    fpath = os.path.join(CLEANED_DIR, fname)
    exists = os.path.exists(fpath)
    size_mb = round(os.path.getsize(fpath) / 1024 / 1024, 1) if exists else 0
    record(f'File exists: {fname}', exists, 
           f'Size: {size_mb} MB' if exists else 'FILE NOT FOUND')

=== CHECK 1: File existence ===
 PASS | File exists: cas_full.csv: Size: 682.2 MB
 PASS | File exists: casualties_clean.csv: Size: 199.7 MB
 PASS | File exists: collisions_clean.csv: Size: 409.7 MB
 PASS | File exists: vehicles_clean.csv: Size: 297.6 MB


---
## Check 2: Load files and check row counts
Loads all four cleaned files into memory and checks that row counts are plausible. The key check is that `cas_full` has the same number of rows as `casualties_clean` — since `cas_full` is a joined version of casualties with collision context, the row count should be identical (one row per casualty).

In [3]:
print('=== CHECK 2: Loading files and row counts ===')
print('Loading files (this may take a minute for large files)...')

cas_full = pd.read_csv(os.path.join(CLEANED_DIR, 'cas_full.csv'),       low_memory=False)
cas      = pd.read_csv(os.path.join(CLEANED_DIR, 'casualties_clean.csv'), low_memory=False)
col      = pd.read_csv(os.path.join(CLEANED_DIR, 'collisions_clean.csv'), low_memory=False)
veh      = pd.read_csv(os.path.join(CLEANED_DIR, 'vehicles_clean.csv'),   low_memory=False)

# Standardise column names
for df in [cas_full, cas, col, veh]:
    df.columns = df.columns.str.lower()

print(f'\nRow counts:')
print(f'  cas_full          : {len(cas_full):>12,}')
print(f'  casualties_clean  : {len(cas):>12,}')
print(f'  collisions_clean  : {len(col):>12,}')
print(f'  vehicles_clean    : {len(veh):>12,}')

# cas_full should have same rows as casualties
counts_match = len(cas_full) == len(cas)
record('Row count: cas_full matches casualties_clean', counts_match,
       f'cas_full={len(cas_full):,} vs casualties={len(cas):,}')

# All files should have >100k rows (sanity check)
for name, df in [('cas_full', cas_full), ('collisions_clean', col), ('vehicles_clean', veh)]:
    record(f'Row count plausible: {name}', len(df) > 100_000,
           f'{len(df):,} rows')

=== CHECK 2: Loading files and row counts ===
Loading files (this may take a minute for large files)...

Row counts:
  cas_full          :    1,748,311
  casualties_clean  :    1,748,311
  collisions_clean  :    1,347,870
  vehicles_clean    :    2,469,085
 PASS | Row count: cas_full matches casualties_clean: cas_full=1,748,311 vs casualties=1,748,311
 PASS | Row count plausible: cas_full: 1,748,311 rows
 PASS | Row count plausible: collisions_clean: 1,347,870 rows
 PASS | Row count plausible: vehicles_clean: 2,469,085 rows


---
## Check 3: Year range
Verifies that the data covers the expected time window (at least 2014–2024) and that there are no unexpected gaps in the year sequence. The client requested 8–10 years of data, so coverage back to at least 2014 is required. Missing years would silently distort trend analysis.

In [4]:
print('=== CHECK 3: Year range ===')

year_min = int(cas_full['collision_year'].min())
year_max = int(cas_full['collision_year'].max())
years    = sorted(cas_full['collision_year'].dropna().unique().astype(int).tolist())

print(f'Year range: {year_min} to {year_max}')
print(f'All years present: {years}')

# Should cover at least 2014-2024
record('Year range: starts at or before 2014', year_min <= 2014, f'Min year = {year_min}')
record('Year range: ends at or after 2024',    year_max >= 2024, f'Max year = {year_max}')

# Check no unexpected year gaps
expected_years = list(range(year_min, year_max + 1))
missing_years  = [y for y in expected_years if y not in years]
record('No unexpected year gaps', len(missing_years) == 0,
       f'Missing years: {missing_years}' if missing_years else 'No gaps found')

=== CHECK 3: Year range ===
Year range: 2014 to 2025
All years present: [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
 PASS | Year range: starts at or before 2014: Min year = 2014
 PASS | Year range: ends at or after 2024: Max year = 2025
 PASS | No unexpected year gaps: No gaps found


---
## Check 4: Duplicate rows
Checks all four files for fully duplicated rows. Duplicates in `cas_full` would inflate KSI counts and distort any LA-level or national trend analysis. This can occur if the join in DataPrep creates a many-to-many match on the accident index.

In [5]:
print('=== CHECK 4: Duplicate rows ===')

for name, df in [('cas_full', cas_full), ('casualties_clean', cas),
                 ('collisions_clean', col), ('vehicles_clean', veh)]:
    n_dups = df.duplicated().sum()
    record(f'No duplicates: {name}', n_dups == 0,
           f'{n_dups:,} duplicate rows found' if n_dups > 0 else '0 duplicates')

=== CHECK 4: Duplicate rows ===
 PASS | No duplicates: cas_full: 0 duplicates
 PASS | No duplicates: casualties_clean: 0 duplicates
 FAIL | No duplicates: collisions_clean: 2,771 duplicate rows found
 PASS | No duplicates: vehicles_clean: 0 duplicates


---
## Check 5: Key columns present and missingness
Confirms that all columns required for the main analysis are present in `cas_full` and reports the percentage of missing values for each. Columns with over 20% missing are flagged. Note: some missingness is expected and documented in STATS19 — for example, deprivation deciles are known to be incomplete.

In [6]:
print('=== CHECK 5: Key columns present and missingness ===')

key_cols = [
    'collision_year',
    'casualty_severity',
    'local_authority_ons_district',
    'speed_limit',
    'urban_or_rural_area',
    'road_type',
    'latitude',
    'longitude'
]

print(f'\n{"Column":<35} {"Present":>8} {"Missing %":>10}')
print('-' * 56)
for c in key_cols:
    present = c in cas_full.columns
    if present:
        pct_missing = cas_full[c].isna().mean() * 100
        flag = ' ⚠ HIGH' if pct_missing > 20 else ''
        print(f'{c:<35} {"Yes":>8} {pct_missing:>9.1f}%{flag}')
        record(f'Column present: {c}', True, f'{pct_missing:.1f}% missing')
    else:
        print(f'{c:<35} {"NO":>8} {"N/A":>10}')
        record(f'Column present: {c}', False, 'COLUMN MISSING FROM cas_full')

=== CHECK 5: Key columns present and missingness ===

Column                               Present  Missing %
--------------------------------------------------------
collision_year                           Yes       0.0%
 PASS | Column present: collision_year: 0.0% missing
casualty_severity                        Yes       0.0%
 PASS | Column present: casualty_severity: 0.0% missing
local_authority_ons_district             Yes       0.0%
 PASS | Column present: local_authority_ons_district: 0.0% missing
speed_limit                              Yes       0.0%
 PASS | Column present: speed_limit: 0.0% missing
urban_or_rural_area                      Yes       0.0%
 PASS | Column present: urban_or_rural_area: 0.0% missing
road_type                                Yes       0.0%
 PASS | Column present: road_type: 0.0% missing
latitude                                 Yes       0.0%
 PASS | Column present: latitude: 0.0% missing
longitude                                Yes       0.0%
 PASS 

---
## Check 6: Severity codes are valid
Confirms that `casualty_severity` only contains the three valid STATS19 codes: 1 (Fatal), 2 (Serious), and 3 (Slight). Any other codes would indicate a data corruption or labelling error. Also checks that the overall KSI percentage falls within a plausible range for GB road casualty data.

In [7]:
print('=== CHECK 6: Severity codes ===')

sev_counts = cas_full['casualty_severity'].value_counts().sort_index()
print('\nCasualty severity value counts:')
print(sev_counts.to_string())

valid_codes = set(sev_counts.index) - {1, 2, 3}
record('Severity codes valid (only 1, 2, 3)', len(valid_codes) == 0,
       f'Invalid codes found: {valid_codes}' if valid_codes else 'Only codes 1, 2, 3 present')

# KSI count sanity check
ksi_count = (cas_full['casualty_severity'].isin([1, 2])).sum()
ksi_pct   = ksi_count / len(cas_full) * 100
print(f'\nKSI casualties: {ksi_count:,} ({ksi_pct:.1f}% of total)')
record('KSI % plausible (5-40%)', 5 <= ksi_pct <= 40,
       f'KSI = {ksi_pct:.1f}% of all casualties')

=== CHECK 6: Severity codes ===

Casualty severity value counts:
casualty_severity
1.0      19316
2.0     278449
3.0    1450546
 PASS | Severity codes valid (only 1, 2, 3): Only codes 1, 2, 3 present

KSI casualties: 297,765 (17.0% of total)
 PASS | KSI % plausible (5-40%): KSI = 17.0% of all casualties


---
## Check 7: Join integrity
Verifies that the join performed in DataPrep correctly added collision-level columns (such as `speed_limit`, `road_type`, `latitude`, `longitude`) to the casualties table. If these columns are missing from `cas_full`, it means the join failed silently and the analysis notebooks will error or produce wrong results.

In [8]:
print('=== CHECK 7: Join integrity ===')

# cas_full should have all columns from casualties plus collision context
cas_cols = set(cas.columns)
full_cols = set(cas_full.columns)
extra_cols = full_cols - cas_cols

print(f'Columns in casualties_clean : {len(cas_cols)}')
print(f'Columns in cas_full         : {len(full_cols)}')
print(f'Extra columns added by join : {len(extra_cols)}')
print(f'\nExtra columns (from collision/vehicle join):')
for c in sorted(extra_cols):
    print(f'  {c}')

# Should have gained collision columns like speed_limit, road_type etc.
expected_joined = ['speed_limit', 'road_type', 'urban_or_rural_area', 'latitude', 'longitude']
for c in expected_joined:
    record(f'Join added column: {c}', c in full_cols,
           'Present' if c in full_cols else 'MISSING — join may have failed')

=== CHECK 7: Join integrity ===
Columns in casualties_clean : 25
Columns in cas_full         : 81
Extra columns added by join : 56

Extra columns (from collision/vehicle join):
  age_band_of_driver
  age_of_driver
  age_of_vehicle
  collision_adjusted_severity_serious
  collision_injury_based
  collision_ref_no_veh
  collision_severity
  collision_year_veh
  date
  day_label
  day_of_week
  driver_distance_banding
  driver_imd_decile
  engine_capacity_cc
  escooter_flag
  fatal
  first_point_of_impact
  force_name
  generic_make_model
  hit_object_in_carriageway
  hit_object_off_carriageway
  journey_purpose_of_driver
  journey_purpose_of_driver_historic
  junction_detail
  junction_label
  junction_location
  ksi
  la_name
  latitude
  light_conditions
  local_authority_ons_district
  longitude
  lsoa_of_driver
  month
  police_force
  propulsion_code
  provisional_veh
  road_surface_conditions
  road_type
  road_type_label
  sex_of_driver
  skidding_and_overturning
  speed_limit
  to

---
## Check 8: Geographic coverage
Checks that the number of unique Local Authorities in `cas_full` is consistent with GB-wide coverage (approximately 300–420 LAs), and that latitude/longitude coordinates fall within the GB bounding box. Coordinates outside GB would indicate either data errors or records from Northern Ireland, which uses a separate reporting system and should not be present.

In [9]:
print('=== CHECK 8: Geographic coverage ===')

n_las = cas_full['local_authority_ons_district'].nunique()
print(f'Unique LAs in cas_full: {n_las}')

# GB has ~380 LAs — should be in that range
record('LA count plausible (300-420)', 300 <= n_las <= 420,
       f'{n_las} unique LAs')

# Coordinate check
valid_coords = (cas_full['latitude'].between(49.5, 61.0) & 
                cas_full['longitude'].between(-8.0, 2.0))
pct_valid = valid_coords.mean() * 100
record('Coordinates within GB bounding box (>95%)', pct_valid > 95,
       f'{pct_valid:.1f}% of coordinates within GB bounds')

=== CHECK 8: Geographic coverage ===
Unique LAs in cas_full: 393
 PASS | LA count plausible (300-420): 393 unique LAs
 PASS | Coordinates within GB bounding box (>95%): 100.0% of coordinates within GB bounds


---
## Check 9: Summary QA report
Collates all check results into a single pass/fail summary table. Any failed checks should be investigated and resolved before proceeding with analysis. The summary is designed to be copied into the portfolio QA sign-off as evidence of data readiness.

In [10]:
print('=== QA SUMMARY REPORT ===')
print()

qa_df = pd.DataFrame(qa_results)
n_pass = (qa_df['Status'] == ' PASS').sum()
n_fail = (qa_df['Status'] == ' FAIL').sum()
n_total = len(qa_df)

print(qa_df.to_string(index=False))
print()
print(f'Total checks : {n_total}')
print(f'Passed       : {n_pass}')
print(f'Failed       : {n_fail}')
print()

if n_fail == 0:
    print(' ALL CHECKS PASSED — data is ready for analysis.')
else:
    print(f' {n_fail} CHECK(S) FAILED — review failed items above before proceeding.')
    print('\nFailed checks:')
    print(qa_df[qa_df['Status'] == ' FAIL'][['Check','Detail']].to_string(index=False))

=== QA SUMMARY REPORT ===

                                       Check Status                                     Detail
                   File exists: cas_full.csv   PASS                             Size: 682.2 MB
           File exists: casualties_clean.csv   PASS                             Size: 199.7 MB
           File exists: collisions_clean.csv   PASS                             Size: 409.7 MB
             File exists: vehicles_clean.csv   PASS                             Size: 297.6 MB
Row count: cas_full matches casualties_clean   PASS cas_full=1,748,311 vs casualties=1,748,311
               Row count plausible: cas_full   PASS                             1,748,311 rows
       Row count plausible: collisions_clean   PASS                             1,347,870 rows
         Row count plausible: vehicles_clean   PASS                             2,469,085 rows
        Year range: starts at or before 2014   PASS                            Min year = 2014
           Year range: 

---
## QA Sign-off
| Item | Detail |
|------|--------|
| **QA run by** | Akeel Shah |
| **Date** | April 2026 |
| **DataPrep notebook version** | ST422_DataPrep.ipynb — run April 2026 |
| **Result** | 31 of 32 checks passed. 1 check failed — see notes. |
| **Notes** | `collisions_clean.csv` contains 2,771 duplicate rows. This is a known issue in the raw DfT STATS19 collision file and does not affect the primary analysis file `cas_full.csv`, which passed all checks with 0 duplicates. All analysis notebooks use `cas_full` directly and are therefore unaffected. No action required. |
---
*This QA notebook is part of the Quality Assurance evidence for ST422 Brief 8.*  
*Source data: DfT STATS19 Open Data — https://www.gov.uk/government/statistical-data-sets/road-safety-open-data*